In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

SEED = 42
TEST_SIZE = 0.2
DATA = "../data/train.csv"
DROP = ["PassengerId", "Name", "Ticket", "Cabin", "SibSp"]

df = pd.read_csv(DATA).drop(columns=DROP)
X, y = df.drop(columns="Survived"), df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

In [ ]:
num_cols = ["Age", "Fare", "Parch", "Pclass"]
cat_cols = ["Sex", "Embarked"]

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh",  OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)),
    ]), cat_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

model = Pipeline([
    ("prep", preprocess),
    ("tree", DecisionTreeClassifier(random_state=SEED)),
])
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))
print(confusion_matrix(y_test, y_pred))
print("Глубина:", model.named_steps["tree"].get_depth(),
      "| листьев:", model.named_steps["tree"].get_n_leaves())

              precision    recall  f1-score   support

           0      0.836     0.882     0.858       110
           1      0.794     0.725     0.758        69

    accuracy                          0.821       179
   macro avg      0.815     0.803     0.808       179
weighted avg      0.820     0.821     0.820       179

ROC-AUC: 0.798
[[97 13]
 [19 50]]
Глубина: 23 | листьев: 158


In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))
from utils import log_result

log_result(
    model_name="Tree",
    y_true=y_test,
    y_proba=y_proba,      
    y_pred=y_pred
)

✓ Tree: AUC=0.7984, acc=0.8212 → c:\Users\АуфПК\VSCode\titanic\results.csv


{'model': 'Tree',
 'timestamp': '2026-11-03 00:29',
 'auc': 0.7984,
 'accuracy': 0.8212,
 'f1': 0.7576,
 'precision': 0.7937,
 'recall': 0.7246,
 'params': '{}',
 'notes': ''}